# Third-party agent simulation

This notebook plays the role of an **external MCP host** (Cursor, Claude, a partner SDK). It sends Knox JWTs to the gateway MCP adapters and never calls Knox, Livy, Hive, or Impala directly.

The runnable example is the documented Spark → Hive path: submit `examples/spark/count_to_10.py`, wait until the batch succeeds, then `hive_select` `{knox_sub}.count_to_10`.

**Before MCP calls:** paste a Knox JWT in the token cell (`getpass`, not echoed). Do not put `KNOX_TOKEN` in AMP project environment or git, and do not print the bearer.

On Compose, an operator staging cell copies the job to HDFS through `/cdp/webhdfs` (not an MCP tool). AMP has no WebHDFS; set `SPARK_FILE_URI` or stage `hdfs:///user/<sub>/examples/count_to_10.py` first.

A LangGraph ReAct agent over the same MCP tools is [`langgraph_agent.ipynb`](langgraph_agent.ipynb) (needs a model API key).

In [ ]:
import json
import os
import sys
from pathlib import Path

# Resolve repo root in Workbench (no __file__ in IPython) or locally.
root = Path(os.environ.get("AGENTGATEWAY_ROOT") or Path.cwd())
if not (root / "pyproject.toml").is_file():
    alt = Path("/home/cdsw")
    if (alt / "pyproject.toml").is_file():
        root = alt
agent_dir = root / "examples" / "agent"
src = root / "src"
for path in (agent_dir, src):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from mcp_agent import (
    health,
    knox_token_status,
    knox_user_from_spark,
    load_knox_token,
    mcp_base_url,
    poll_spark_batch,
    profile,
    require_tool_ok,
    spark_job_uri,
    submit_spark_example,
    tools_call,
    tools_list,
)

print("profile:", profile())
print("spark url:", mcp_base_url("spark"))
print("hive url:", mcp_base_url("hive"))
print("impala url:", mcp_base_url("impala"))

## Knox JWT (this session only)

AMP project env is for `KNOX_PROXY_URL`, not the user bearer. Paste a **Knox Token API JWT** (three segments, usually starts with `eyJ`). Do not paste a Knox passcode, cookie, or `Bearer ` prefix.

The cell stores it in `os.environ` for this engine only and never prints the bearer. It does print `alg`, `iss`, `sub`, and seconds until `exp`. `401 invalid_token` means the gateway could not parse the value as a JWT.

In [ ]:
token = load_knox_token(prompt=True, ignore_non_jwt_env=True)
status = knox_token_status(token)
print("knox jwt:", "set" if status["set"] else "missing")
print("jwt_shaped:", status["jwt_shaped"])
print("alg:", status["alg"] or "-")
print("iss:", status["iss"] or "-")
print("sub:", status["sub"] or "-")
print("exp_in_s:", status["exp_in_s"])
print("hint:", status["hint"])
if not status["jwt_shaped"] or status["hint"] != "ok":
    raise RuntimeError(status["hint"] or "Knox JWT is not usable")

## Public health (no JWT)

MCP applications expose `GET /health` without authentication. This confirms the adapter is listening before we send a bearer.

In [ ]:
for adapter in ("spark", "hive", "impala"):
    try:
        print(adapter, health(adapter))
    except Exception as exc:
        print(adapter, "unavailable:", exc)

## Discover tools (`tools/list`)

Real MCP hosts call `tools/list` after `initialize`. We list Spark, Hive, and Impala tool catalogs.

In [ ]:
for adapter in ("spark", "hive", "impala"):
    try:
        names = sorted(t["name"] for t in tools_list(adapter))
        print(f"{adapter}:", ", ".join(names))
    except Exception as exc:
        print(f"{adapter}: skipped ({exc})")

## Spark → Hive example

1. Read the Knox `sub` from Spark MCP (`spark_list_batches`).
2. Submit `count_to_10.py` with `spark_submit_batch` (write as that subject), or **reuse** an in-flight/success `count-to-10` batch so a re-run does not start a second Livy application. Set `SPARK_FORCE_SUBMIT=1` to submit again.
3. Poll `spark_get_batch` until `success` or `dead`.
4. Query Iceberg `{sub}.count_to_10` with Hive (`hive_list_tables`, `hive_describe_table`, `hive_select`).

Named columns only; `limit` ≤ 50. No `SELECT *`, no DDL from Hive.

In [ ]:
knox_user = knox_user_from_spark()
file_uri = spark_job_uri(knox_user)
database = knox_user.split("@", 1)[0]
table = os.environ.get("HIVE_EXAMPLE_TABLE", "count_to_10")
print("knox_user:", knox_user)
print("file:", file_uri)
print("hive target:", f"{database}.{table}")

### Operator staging (Compose only)

Agents do not call `/cdp/webhdfs`. This cell is the operator step from [examples/spark/README.md](../spark/README.md) so a local `gateway up` can submit a real URI. AMP skips it.

In [ ]:
if profile() != "compose":
    print("AMP: WebHDFS is not an agent route. Using", file_uri)
else:
    from agentgateway.webhdfs import put_file

    token = (os.environ.get("KNOX_TOKEN") or "").strip()
    if not token:
        raise RuntimeError("Set KNOX_TOKEN before staging")
    local = root / "examples" / "spark" / "count_to_10.py"
    hdfs_path = "/" + file_uri.split(":///", 1)[-1].lstrip("/")
    status = put_file(local, hdfs_path, token)
    meta = status.get("FileStatus") or status
    print("staged", hdfs_path, "length", meta.get("length"))

### Submit and poll Spark

`spark_submit_batch` is a write as the Knox subject. Ranger must allow Livy for that user. The job reuses Livy's SparkSession (it does not `spark.stop()`). Re-running this cell reuses an existing `count-to-10` batch unless `SPARK_FORCE_SUBMIT=1`. Poll until `state` is `success` (then Hive) or `dead` (inspect `spark_get_log`).

In [ ]:
submit = submit_spark_example(file_uri=file_uri, name="count-to-10")
batch_id = submit.get("id")
print("reused:" if submit.get("reused") else "submitted:", json.dumps(submit, indent=2))
if batch_id is None:
    raise RuntimeError("spark_submit_batch did not return an id")

batch = poll_spark_batch(int(batch_id))
print("final state:", batch.get("state"), "id:", batch.get("id"))
if str(batch.get("state") or "").lower() != "success":
    log = require_tool_ok(tools_call("spark_get_log", {"batch_id": int(batch_id)}, adapter="spark"))
    lines = log.get("log") or []
    print("log tail:")
    print("\n".join(lines[-20:] if isinstance(lines, list) else [str(lines)]))
    raise RuntimeError(f"batch {batch_id} ended {batch.get('state')!r}; Hive select would fail")

### Hive select `{sub}.count_to_10`

Same Knox JWT. Expected rows: `n` = 1 … 10.

In [ ]:
tables = require_tool_ok(
    tools_call("hive_list_tables", {"database": database}, adapter="hive", timeout=120.0)
)
print("tables:", tables.get("items") or tables.get("tables") or tables)

described = require_tool_ok(
    tools_call(
        "hive_describe_table",
        {"database": database, "table": table},
        adapter="hive",
        timeout=120.0,
    )
)
print("describe:", json.dumps(described.get("columns") or described, indent=2))

selected = require_tool_ok(
    tools_call(
        "hive_select",
        {"database": database, "table": table, "columns": "n", "limit": 10},
        adapter="hive",
        timeout=120.0,
    )
)
print("returned:", selected.get("returned"), "truncated:", selected.get("truncated"))
print(json.dumps(selected.get("rows"), indent=2))